In [ ]:
# Cell 1: Environment Setup
!pip install shap -q

import os
import cv2
import shap
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from google.colab import files

# Hardcoded Config for Portability
CLASS_NAMES = ["LPD", "PD"] # 0 = Healthy, 1 = Dysgraphia
MODEL_PATH = "FINAL_production_model.keras"

print("✓ Environment Ready!")

In [ ]:
# Cell 2: The Diagnostic Engine

def make_gradcam_heatmap(img_array, model, last_conv_layer_name=None, pred_index=None):
    """Sledgehammer override to bypass Keras 3 graph bugs."""
    if len(img_array.shape) == 3: img_array = np.expand_dims(img_array, axis=0)
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
    x = tf.concat([img_tensor, img_tensor, img_tensor], axis=-1)

    base_model = model.get_layer("MobileNetV3Small")
    pool_layer = model.get_layer("global_avg_pool")
    bn_layer = model.get_layer("batch_norm")
    dense_layer = model.get_layer("dense_128")
    dropout_layer = model.get_layer("dropout")
    classifier = model.get_layer("classifier")

    with tf.GradientTape() as tape:
        last_conv_layer_output = base_model(x, training=False)
        tape.watch(last_conv_layer_output)

        x_head = pool_layer(last_conv_layer_output)
        x_head = bn_layer(x_head, training=False)
        x_head = dense_layer(x_head)
        x_head = dropout_layer(x_head, training=False)
        preds = classifier(x_head)

        if pred_index is None: pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = (tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)).numpy()

    heatmap_resized = cv2.resize(heatmap, (img_array.shape[2], img_array.shape[1]))

    return heatmap_resized

def generate_clinical_narrative(img_array, shap_values, predicted_class, confidence):
    """Deterministically generates a clinical summary supported by neurodevelopmental literature."""
    img_squeezed = np.squeeze(img_array)
    img_uint8 = np.uint8(255 * img_squeezed) if img_squeezed.max() <= 1.0 else np.uint8(img_squeezed)

    shap_val = shap_values.values[0, ..., 0] if hasattr(shap_values, "values") else shap_values[0]
    if len(shap_val.shape) == 3: shap_val = np.sum(shap_val, axis=-1) if shap_val.shape[-1] == 3 else np.squeeze(shap_val, axis=-1)

    ink_mask = img_uint8 < 200
    row_has_ink = np.any(ink_mask, axis=1)
    col_has_ink = np.any(ink_mask, axis=0)

    if not np.any(row_has_ink): return "System Warning: Unable to detect sufficient handwriting for spatial analysis."

    min_y, max_y = np.argmax(row_has_ink), len(row_has_ink) - np.argmax(row_has_ink[::-1])
    min_x, max_x = np.argmax(col_has_ink), len(col_has_ink) - np.argmax(col_has_ink[::-1])

    zone1_morphology = ink_mask.copy()
    zone2_kerning = np.zeros_like(ink_mask, dtype=bool)
    zone2_kerning[min_y:max_y, min_x:max_x] = ~ink_mask[min_y:max_y, min_x:max_x]
    zone3_spatial = np.ones_like(ink_mask, dtype=bool)
    zone3_spatial[min_y:max_y, :] = False

    supportive_shap = np.maximum(shap_val, 0)
    score_z1 = np.sum(supportive_shap[zone1_morphology])
    score_z2 = np.sum(supportive_shap[zone2_kerning])
    score_z3 = np.sum(supportive_shap[zone3_spatial])

    total_score = score_z1 + score_z2 + score_z3 or 1e-9
    pct_z1, pct_z2, pct_z3 = score_z1 / total_score, score_z2 / total_score, score_z3 / total_score

    narrative = f"▶ AIKONIC CLINICAL DIAGNOSTIC REPORT\nClassification: {predicted_class} ({confidence:.1f}% System Confidence)\n" + "-" * 60 + "\n"

    if predicted_class == "PD":
        narrative += "EVIDENCE-BASED FINDINGS:\n"
        if pct_z3 > max(pct_z1, pct_z2):
            narrative += "The highest predictive anomalies are located in the upper or lower boundary margins of the handwriting sample. According to Deuel (1995), this pattern is characteristic of Spatial Dysgraphia, a subtype defined by impaired understanding of space that results in an inability to adhere to baselines and respect page margins regardless of letter-formation ability. This constitutes impaired macro-spatial planning, a primary indicator of the visual-spatial deficits associated with Specific Learning Disorder in written expression (American Psychiatric Association [APA], 2013). The Beery-Buktenica Developmental Test of Visual-Motor Integration (Beery VMI) — one of the most widely used standardized occupational therapy instruments — explicitly assesses a patient's ability to stay within designated spatial boundaries; the anomalies flagged in this zone represent the automated equivalent of that assessment criterion (Beery & Beery, 2010)."
        elif pct_z2 > max(pct_z1, pct_z3):
            narrative += "The predictive anomalies are heavily localized in the whitespace between characters. Deuel (1995) defines Spatial Dysgraphia as producing illegible writing — whether spontaneous or copied — due to a fundamental deficit in spatial perception, which manifests directly as abnormal letter spacing and erratic kerning. Chung et al. (2020) further specify that in spatial dysgraphia, oral spelling and fine-motor tapping speed are preserved, indicating that the spacing irregularities detected in this zone are perceptual-spatial in origin rather than purely motoric. These atypical inter-character intervals are a strong behavioral marker of impaired graphomotor coordination, consistent with Specific Learning Disorder criteria (APA, 2013). Döhla and Heim (2016) additionally note that dysgraphia and dyslexia share spatial-processing deficits that manifest during the physical execution of written output."
        else:
            narrative += "The predictive anomalies are concentrated directly on the ink strokes themselves. Deuel (1995) classifies this presentation as Motor Dysgraphia, a subtype in which both spontaneous writing and copying are impaired due to deficient fine-motor coordination and graphomotor execution — distinguishable from other subtypes by abnormal finger-tapping speed and drawing performance. This is consistent with DSM-5 criteria for Specific Learning Disorder with impairment in written expression, which includes difficulties with the clarity and physical organization of written output (APA, 2013). Chung et al. (2020) describe this subtype as reflecting an inefficiency in the graphomotor loop, wherein motor memory fails to produce consistent letter formation. Döhla and Heim (2016) further associate motor dysgraphia with deficits in the automatization of fine-motor writing sequences, resulting in erratic pen pressure and inconsistent letterform."
    else:
        narrative += "EVIDENCE-BASED FINDINGS:\nAnalysis indicates that graphomotor execution falls within healthy neurodevelopmental limits, with no dominant zone of anomalous SHAP attribution corresponding to known dysgraphia subtypes (Deuel, 1995; APA, 2013). "
        if pct_z1 > 0.5: narrative += "SHAP attributions are primarily distributed across the ink strokes (Zone 1), with letter morphology, line stability, and fine-motor execution consistent with standard age-appropriate motor development, as defined by Motor Dysgraphia absence criteria (Deuel, 1995)."
        elif pct_z3 > 0.5: narrative += "SHAP attributions are primarily distributed in the margin regions (Zone 3). Proper utilization of margins and baseline adherence indicate healthy visual-spatial planning and cognitive organization, consistent with passing performance on spatial boundary tasks assessed by the Beery VMI (Beery & Beery, 2010)."
        else: narrative += "Macro-spatial planning (Zone 3), intra-word kerning (Zone 2), and letter morphology (Zone 1) all appear balanced and within normal developmental thresholds for written expression, consistent with the absence of Spatial or Motor Dysgraphia indicators described by Deuel (1995) and Mather and Wendling (2011)."
    return narrative

def plot_4_panel_diagnostic(img_array, heatmap, shap_values, predicted_class, save_name="output_report.png"):
    """Renders the Color-Locked, Bubble-Masked UI."""
    img_squeezed = np.squeeze(img_array)
    img_uint8 = np.uint8(255 * img_squeezed) if img_squeezed.max() <= 1.0 else np.uint8(img_squeezed)
    img_rgb = cv2.cvtColor(img_uint8, cv2.COLORMAP_BONE) if len(img_uint8.shape) == 2 else img_uint8

    ink_mask_uint8 = np.uint8((img_uint8 < 200) * 255)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (40, 40))
    dilated_mask = cv2.dilate(ink_mask_uint8, kernel)
    spatial_mask_224 = cv2.resize(dilated_mask, (img_uint8.shape[1], img_uint8.shape[0])) > 0

    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    gradcam_overlay = np.where(spatial_mask_224[..., None], cv2.addWeighted(img_rgb, 0.5, heatmap_color, 0.5, 0), img_rgb)

    shap_val = shap_values.values[0, ..., 0] if hasattr(shap_values, 'values') else shap_values[0]
    if len(shap_val.shape) == 3: shap_val = np.sum(shap_val, axis=-1) if shap_val.shape[-1] == 3 else np.squeeze(shap_val, axis=-1)

    if predicted_class == "LPD": shap_val *= -1 # Color Lock

    spatial_mask_shap = cv2.resize(dilated_mask, (shap_val.shape[1], shap_val.shape[0])) > 0
    signal_mask = np.abs(shap_val) >= np.percentile(np.abs(shap_val), 65)
    masked_shap = np.ma.masked_where(~(spatial_mask_shap & signal_mask), shap_val)

    focus_mask = heatmap_resized > np.percentile(heatmap_resized, 70)
    focus_heatmap = np.zeros_like(heatmap_color)
    focus_heatmap[focus_mask] = heatmap_color[focus_mask]
    focus_overlay = np.where(spatial_mask_224[..., None], cv2.addWeighted(img_rgb, 0.7, focus_heatmap, 0.5, 0), img_rgb)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle('AIKONIC Dysgraphia Diagnostic Analysis', fontsize=16, fontweight='bold', y=1.05)

    axes[0].imshow(img_uint8, cmap='gray'); axes[0].set_title('1. Original Sample')
    axes[1].imshow(cv2.cvtColor(gradcam_overlay, cv2.COLOR_BGR2RGB)); axes[1].set_title('2. Grad-CAM Localization')
    axes[2].imshow(img_uint8, cmap='gray'); axes[2].imshow(masked_shap, cmap='coolwarm', alpha=0.75, vmin=-np.max(np.abs(shap_val)), vmax=np.max(np.abs(shap_val))); axes[2].set_title('3. SHAP Attribution')
    axes[3].imshow(cv2.cvtColor(focus_overlay, cv2.COLOR_BGR2RGB)); axes[3].set_title('4. Severe Anomaly Focus')

    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()

print("✓ Production Diagnostic Engine Loaded!")

In [ ]:
# Cell 3: Master Pipeline (Vertical UI)

import cv2
import numpy as np
import shap
import time
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files

def standardize_patch_and_track(img_path: str, target_size: int = 224, overlap: float = 0.5):
    """
    Exact preprocessing from training, but tracks spatial metadata
    so we can perfectly reverse-engineer the heatmaps back to full-page.
    """
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None: raise ValueError(f"Could not load image: {img_path}")

    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if img.mean() < 128: img = cv2.bitwise_not(img)

    h, w = img.shape
    image_ar = w / h
    patches = []
    metadata = [] # Tracks how to "undo" the patching

    if image_ar <= 2.5: # SCENARIO A: Multi-line
        scale = target_size / h
        new_w = int(w * scale)
        resized = cv2.resize(img, (new_w, target_size), interpolation=cv2.INTER_LINEAR)
        step = int(target_size * (1 - overlap))

        if new_w <= target_size:
            canvas = np.full((target_size, target_size), 255, dtype=np.uint8)
            offset_x = (target_size - new_w) // 2
            canvas[:, offset_x : offset_x + new_w] = resized
            patches.append(canvas)
            metadata.append({"type": "A_small", "offset_x": offset_x, "new_w": new_w, "orig_w": w, "orig_h": h})
        else:
            for x in range(0, new_w - target_size + 1, step):
                patches.append(resized[:, x : x + target_size])
                metadata.append({"type": "A_slide", "x_start": x, "scale": scale, "orig_w": w, "orig_h": h})
            if (new_w - target_size) % step != 0:
                patches.append(resized[:, new_w - target_size : new_w])
                metadata.append({"type": "A_slide", "x_start": new_w - target_size, "scale": scale, "orig_w": w, "orig_h": h})
    else: # SCENARIO B: Single-line
        chunk_w = int(h * 3.0)
        step = int(chunk_w * (1 - overlap))

        def process_chunk(x_start, c_width):
            crop = img[:, x_start : x_start + c_width]
            scale = target_size / c_width
            new_h = int(h * scale)
            resized_crop = cv2.resize(crop, (target_size, new_h), interpolation=cv2.INTER_LINEAR)
            canvas = np.full((target_size, target_size), 255, dtype=np.uint8)
            offset_y = (target_size - new_h) // 2
            canvas[offset_y : offset_y + new_h, :] = resized_crop
            patches.append(canvas)
            metadata.append({"type": "B", "x_start": x_start, "c_width": c_width, "offset_y": offset_y, "new_h": new_h, "orig_h": h})

        for x in range(0, w - chunk_w + 1, step):
            process_chunk(x, chunk_w)
        if (w - chunk_w) % step != 0 and w > chunk_w:
            process_chunk(w - chunk_w, chunk_w)

    return img, patches, metadata


def process_and_plot_master(image_path, model):
    print(f"\n{'='*80}\n▶ PROCESSING: {Path(image_path).name}\n{'='*80}")
    start_time = time.time()

    # 1. Preprocessing & Tracking
    orig_img, patches, meta = standardize_patch_and_track(image_path)
    if not patches: return print("Error: No patches generated.")

    h, w = orig_img.shape
    full_gradcam = np.zeros((h, w), dtype=np.float32)
    full_shap = np.zeros((h, w), dtype=np.float32)
    overlap_count = np.zeros((h, w), dtype=np.float32)

    # 2. Probability Averaging & Detailed Breakdown
    pd_probs = []
    patch_inputs = []

    for patch_img in patches:
        patch_input = np.expand_dims(np.expand_dims(patch_img.astype('float32') / 255.0, -1), 0)
        patch_inputs.append(patch_input)
        preds = model.predict(patch_input, verbose=0)
        pd_probs.append(preds[0][1])

    avg_pd_prob = np.mean(pd_probs)
    final_diagnosis = "PD" if avg_pd_prob >= 0.5 else "LPD"
    page_conf = avg_pd_prob * 100 if final_diagnosis == "PD" else (1 - avg_pd_prob) * 100

    print(f"   ✓ Extracted {len(patches)} standardized patches.\n")
    print("   [Patch-Level Confidence Breakdown]")

    # Dynamically build the breakdown and the math string
    math_str_parts = []
    for i, prob in enumerate(pd_probs):
        prob_pct = prob * 100

        # Calculate True Confidence and assign the clinical label
        if prob >= 0.5:
            conf_pct = prob_pct
            diag_label = "Dysgraphic (PD)"
        else:
            conf_pct = (1.0 - prob) * 100
            diag_label = "Normal (LPD)"

        # Display the raw probability first, followed by the model's confidence
        print(f"   • Patch {i+1} ({prob_pct:>4.1f}% PD): The model is {conf_pct:>4.1f}% confident this is {diag_label}.")
        math_str_parts.append(f"{prob_pct:.1f}")

    # Format the mathematical synthesis string
    math_str = " + ".join(math_str_parts)
    if len(patches) > 8:
        math_str = math_str[:40] + "... " # Truncate visually for massive documents

    print("\n   [Page-Level Synthesis]")
    print(f"   • Mathematical Average : ({math_str}) / {len(patches)}")
    print(f"   • Average PD Probability: {avg_pd_prob*100:.1f}%")
    print(f"   ▶ PAGE DIAGNOSIS: {final_diagnosis} ({page_conf:.1f}% Overall System Confidence)")

    # 3. Inverse Spatial Mapping
    print("\n   Generating Diagnostics & Stitching Full Page...")
    masker = shap.maskers.Image("inpaint_telea", (224, 224, 1))
    explainer = shap.Explainer(model.predict, masker, output_names=CLASS_NAMES)

    for i, (p_input, m) in enumerate(zip(patch_inputs, meta)):
        hm = make_gradcam_heatmap(p_input, model)
        sv = explainer(p_input, max_evals=500, outputs=shap.Explanation.argsort.flip[:1])
        shap_val = sv.values[0, ..., 0]
        if len(shap_val.shape) == 3: shap_val = np.sum(shap_val, axis=-1)
        if final_diagnosis == "LPD": shap_val *= -1

        # Reverse Engineering Padding/Scaling
        if m["type"] == "B":
            hm_cropped = hm[m["offset_y"] : m["offset_y"] + m["new_h"], :]
            shap_cropped = shap_val[m["offset_y"] : m["offset_y"] + m["new_h"], :]
            hm_restored = cv2.resize(hm_cropped, (m["c_width"], m["orig_h"]))
            shap_restored = cv2.resize(shap_cropped, (m["c_width"], m["orig_h"]))
            x_s, x_e = m["x_start"], m["x_start"] + m["c_width"]
            full_gradcam[:, x_s:x_e] += hm_restored
            full_shap[:, x_s:x_e] += shap_restored
            overlap_count[:, x_s:x_e] += 1

        elif m["type"] == "A_small":
            hm_cropped = hm[:, m["offset_x"] : m["offset_x"] + m["new_w"]]
            shap_cropped = shap_val[:, m["offset_x"] : m["offset_x"] + m["new_w"]]
            full_gradcam += cv2.resize(hm_cropped, (w, h))
            full_shap += cv2.resize(shap_cropped, (w, h))
            overlap_count += 1

        elif m["type"] == "A_slide":
            hm_restored = cv2.resize(hm, (int(224 / m["scale"]), h))
            shap_restored = cv2.resize(shap_val, (int(224 / m["scale"]), h))
            x_s = int(m["x_start"] / m["scale"])
            x_e = x_s + hm_restored.shape[1]
            if x_e > w: hm_restored, shap_restored, x_e = hm_restored[:, :w-x_s], shap_restored[:, :w-x_s], w
            full_gradcam[:, x_s:x_e] += hm_restored
            full_shap[:, x_s:x_e] += shap_restored
            overlap_count[:, x_s:x_e] += 1

    mask = overlap_count > 0
    full_gradcam[mask] /= overlap_count[mask]
    full_shap[mask] /= overlap_count[mask]

    # 4. Narrative
    narrative = generate_clinical_narrative(orig_img, [full_shap], final_diagnosis, page_conf)
    print("\n" + "="*80 + "\n" + "▶ AIKONIC CLINICAL DIAGNOSTIC REPORT\n" + "="*80)
    print(narrative)
    print("="*80 + "\n")

    # 5. DYNAMIC VERTICAL UI RENDERING (The Whitespace Fix)
    img_rgb = cv2.cvtColor(orig_img, cv2.COLOR_GRAY2RGB)
    ink_mask = cv2.bitwise_not(orig_img)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (40, 40))
    dilated_mask = cv2.dilate(ink_mask, kernel) > 0

    heatmap_color = cv2.applyColorMap(np.uint8(255 * full_gradcam), cv2.COLORMAP_JET)
    gradcam_overlay = np.where(dilated_mask[..., None], cv2.addWeighted(img_rgb, 0.5, heatmap_color, 0.5, 0), img_rgb)

    signal_mask = np.abs(full_shap) >= np.percentile(np.abs(full_shap[mask]), 65) if np.any(mask) else np.zeros_like(full_shap, dtype=bool)
    masked_shap = np.ma.masked_where(~(dilated_mask & signal_mask), full_shap)

    f_mask = full_gradcam > np.percentile(full_gradcam[mask], 70) if np.any(mask) else np.zeros_like(full_gradcam, dtype=bool)
    f_heat = np.zeros_like(heatmap_color)
    f_heat[f_mask] = heatmap_color[f_mask]
    focus_overlay = np.where(dilated_mask[..., None], cv2.addWeighted(img_rgb, 0.7, f_heat, 0.5, 0), img_rgb)

    # ── DYNAMIC FIGSIZE CALCULATION ──
    aspect_ratio = h / w
    base_width = 16 # Keep it wide for high readability
    panel_height = base_width * aspect_ratio

    # 4 panels + 2 inches extra for the main title and sub-titles
    total_height = (panel_height * 4) + 2.5

    fig, axes = plt.subplots(4, 1, figsize=(base_width, total_height))
    fig.suptitle('AIKONIC Full-Page Diagnostic Scan', fontsize=20, fontweight='bold', y=0.98)

    axes[0].imshow(orig_img, cmap='gray')
    axes[0].set_title('1. Standardized Original Document', fontsize=16, pad=10)

    axes[1].imshow(cv2.cvtColor(gradcam_overlay, cv2.COLOR_BGR2RGB))
    axes[1].set_title('2. Global Localization (Grad-CAM)', fontsize=16, pad=10)

    vmax = np.max(np.abs(full_shap[mask])) if np.any(mask) else 1
    axes[2].imshow(orig_img, cmap='gray')
    axes[2].imshow(masked_shap, cmap='coolwarm', alpha=0.75, vmin=-vmax, vmax=vmax)
    axes[2].set_title('3. Global SHAP Attribution (Feature Importance)', fontsize=16, pad=10)

    axes[3].imshow(cv2.cvtColor(focus_overlay, cv2.COLOR_BGR2RGB))
    axes[3].set_title('4. Severe Anomaly Focus', fontsize=16, pad=10)

    for ax in axes:
        ax.axis('off')

    # h_pad forces the titles to sit closer to the images
    plt.tight_layout(h_pad=1.5)
    save_name = f"vertical_master_report_{Path(image_path).name}.png"
    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()

    exec_time = time.time() - start_time
    print(f"Total Processing Time: {exec_time:.2f} seconds")


# The Upload Trigger
import os

if not os.path.exists(MODEL_PATH):
    print(f"PLEASE UPLOAD YOUR MODEL FIRST: {MODEL_PATH}")
else:
    if 'prod_model' not in locals():
        import tensorflow as tf
        prod_model = tf.keras.models.load_model(MODEL_PATH)

    print("Upload a full handwriting sample page (.jpg or .png):")
    uploaded = files.upload()
    for filename in uploaded.keys():
        process_and_plot_master(filename, prod_model)